# 15b · Rebuilding the Earth Engine layers from scratch (part 2)

Needs a current Earth Engine login (`gcloud auth application-default login`); it fails loudly without one. Run
after notebook 15, which downloads the inputs into the fresh folder.

Rebuilt with this folder's code only: all six North Dakota tiles (AlphaEarth and CDL from Earth Engine, NWI from
the fresh state download), NC4 (North Carolina) and MN1 (held out), each with every array compared for exact
equality with the tile every result was computed from.

In [1]:
import json
import time

import numpy as np

from askmap.config import PROJECT_ROOT, data_root, load_config, load_env
from askmap.data import read_uspvdb
from askmap.earth_engine import init_earth_engine
from askmap.manifest import Manifest
from askmap.tiles import area_tiles, build_ee_tile, build_nwi_tile, save_tile

cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
OLD = data_root()
NEW = OLD.parent / "ask_the_map_rebuild_check"
RES = PROJECT_ROOT / "outputs" / "results"
part1 = json.loads((RES / "15_rebuild_check.json").read_text())  # notebook 15 must have run first
load_env()
YEAR, U = cfg["land_year"], cfg["uspvdb"]
nd = cfg["areas"]["north_dakota"]
nwi_zip = NEW / nd["nwi_zip"]
pv_zip = NEW / "raw" / "uspvdbGeoJSON.zip"
raw = Manifest(NEW / "raw")
if not (raw.is_current(nwi_zip) and raw.is_current(pv_zip)):
    raise RuntimeError("run notebook 15 first: the fresh downloads are missing")
_ = init_earth_engine()  # the project ID is read from .env and not printed
T_START = time.time()

/Volumes/disco2/github/climate-embeddings-crop-integrity/ask-the-map/.venv/lib/python3.12/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


/Volumes/disco2/github/climate-embeddings-crop-integrity/ask-the-map/.venv/lib/python3.12/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


In [2]:
pv = read_uspvdb(pv_zip, U["member"])
want = {"north_dakota": None, "north_carolina": ["NC4"], "minnesota": ["MN1"]}
tile_dir = NEW / "tiles"
tile_dir.mkdir(exist_ok=True)
man = Manifest(tile_dir)
rows = []
for area, names in want.items():
    acfg = cfg["areas"][area]
    for name, (tile, _role) in area_tiles(acfg, cfg["grid"]).items():
        if names is not None and name not in names:
            continue
        t0 = time.time()
        if acfg["source"] == "earth_engine_nwi":
            arrays = build_nwi_tile(tile, YEAR, f"zip://{nwi_zip}!{acfg['nwi_gdb']}", acfg["nwi_layer"])
        else:
            arrays = build_ee_tile(tile, YEAR, pv, U["last_year"])
        path = tile_dir / f"{area}_{name}.npz"
        save_tile(path, arrays, man, area=area, tile=name, year=YEAR, source=acfg["source"])
        old = np.load(OLD / "tiles" / f"{area}_{name}.npz")
        row = {
            "area": area,
            "tile": name,
            "seconds": round(time.time() - t0, 1),
            "same_keys": sorted(old.files) == sorted(arrays),
        }
        for k in old.files:
            a, b = old[k], arrays[k]
            same = a.shape == b.shape and a.dtype == b.dtype and np.array_equal(a, b, equal_nan=True)
            row[f"{k}_identical"] = bool(same)
            if not same and a.shape == b.shape:
                fa, fb = a.astype(float), b.astype(float)
                diff = ~((fa == fb) | (np.isnan(fa) & np.isnan(fb)))
                row[f"{k}_share_different"] = float(diff.mean())
                row[f"{k}_max_abs_diff"] = float(np.nanmax(np.abs(fa - fb)))
        rows.append(row)
        print(row)

/Volumes/disco2/github/climate-embeddings-crop-integrity/ask-the-map/.venv/lib/python3.12/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


{'area': 'north_dakota', 'tile': 'Stutsman', 'seconds': 29.7, 'same_keys': True, 'ae_identical': True, 'cdl_identical': True, 'nwi_identical': True}


/Volumes/disco2/github/climate-embeddings-crop-integrity/ask-the-map/.venv/lib/python3.12/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


{'area': 'north_dakota', 'tile': 'Barnes', 'seconds': 33.9, 'same_keys': True, 'ae_identical': True, 'cdl_identical': True, 'nwi_identical': True}


/Volumes/disco2/github/climate-embeddings-crop-integrity/ask-the-map/.venv/lib/python3.12/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


...

/Volumes/disco2/github/climate-embeddings-crop-integrity/ask-the-map/.venv/lib/python3.12/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


{'area': 'north_dakota', 'tile': 'Eddy', 'seconds': 30.3, 'same_keys': True, 'ae_identical': True, 'cdl_identical': True, 'nwi_identical': True}


{'area': 'north_carolina', 'tile': 'NC4', 'seconds': 28.1, 'same_keys': True, 'ae_identical': True, 'cdl_identical': True, 'solar_identical': True}


/Volumes/disco2/github/climate-embeddings-crop-integrity/ask-the-map/.venv/lib/python3.12/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


{'area': 'minnesota', 'tile': 'MN1', 'seconds': 26.8, 'same_keys': True, 'ae_identical': True, 'cdl_identical': True, 'solar_identical': True}


In [3]:
ok = all(r["same_keys"] and all(v for k, v in r.items() if k.endswith("_identical")) for r in rows)
check = {"tiles": rows, "tiles_identical": ok, "minutes": round((time.time() - T_START) / 60, 1)}
(RES / "15b_rebuild_check_ee.json").write_text(json.dumps(check, indent=1, default=str))
print(f"Earth Engine tiles identical: {ok}; {check['minutes']} min")

Earth Engine tiles identical: True; 4.9 min
